<div style="
    background:linear-gradient(135deg,#9941d8,#6d5dfc);
    color:white;
    padding:25px;
    border-radius:15px;
">
<h1 style="margin:0;">Restaurant Reservation System 🍽️</h1>
<p style="margin-top:10px;font-size:18px;">
An Agent-Based Restaurant Reservation Chatbot using LangChain
</p>
</div>



<div style="text-align: center;">
<img src='./assets/header.png' alt="header" width=70% >
</div>




<div style="
    border-left:6px solid #9941d8;
    padding:18px;
    border-radius:10px;
    margin-top:15px;
">

<h3 style="color:#9941d8;margin-top:0;">ℹ️ Project Information</h3>

<b>👤 Author:</b> MohammadReza Babagoli<br>
<b>📅 Date:</b> 2026<br>
<b>🔗 Contact:</b>
<a href="https://www.linkedin.com/in/mohammadreza-babagoli" target="_blank">
    <img src="https://img.shields.io/badge/LinkedIn-Profile-0A66C2?style=flat&logo=linkedin&logoColor=white">
</a>

<hr>

<h3 style="color:#9941d8;">🎯 Objective</h3>

This notebook demonstrates an <b>agent-based restaurant reservation chatbot</b> capable of:
<ul>
    <li>🔍 Searching restaurants</li>
    <li>📅 Making reservations</li>
    <li>❌ Canceling bookings</li>
    <li>❓ Answering restaurant FAQs</li>
</ul>

</div>

<div style="text-align: center;">
<img src='./assets/system_architecture.png' alt="system architechture" width=80%>
</div>


## Required Packages

This notebook was developed and tested with **Python 3.13+**.

Install the required packages using:

```bash
pip install pandas python-dotenv langchain langgraph dateparser phonenumbers pillow qrcode langchain-openai langchain-chroma openpyxl
```

### Main Dependencies

| Package | Purpose |
|---------|---------|
| pandas | Read and manipulate tabular data |
| python-dotenv | Load environment variables from `.env` |
| langchain | Build tools and AI agents |
| langgraph | Agent workflow and state management |
| dateparser | Parse natural language dates and times |
| phonenumbers | Parse and validate phone numbers |
| Pillow | Create and edit images (receipts) |
| qrcode | Generate QR codes |
| langchain-openai | OpenAI-compatible embeddings and models |
| openpyxl | Read and write Excel (`.xlsx`) files |

In [ ]:
# !pip install pandas python-dotenv langchain langgraph dateparser phonenumbers pillow qrcode langchain-openai langchain-chroma openpyxl

## Libraries

In [3]:
import pandas as pd
from typing import Optional
from dotenv import load_dotenv
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver
import random
import string
from datetime import datetime
from zoneinfo import ZoneInfo
import dateparser
import phonenumbers
from PIL import Image, ImageDraw, ImageFont
import qrcode
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
import os 

## Configuration (API keys, model settings).

### Load OPENROUTER API Keys

In [4]:
# This command will automatically load your API key from the .env file.
load_dotenv()  # OPENROUTER_API_KEY=sk-or-v1-...

True

### LLM Model:

In [ ]:
# specify your LLM model in the ##Run section for the agent.

### Load Embedding model

In [5]:
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

## Loading Data

In [6]:
"""
The restaurant information is loaded from an Excel file.
Each row represents one restaurant together with its cuisine, city,
rating, and available reservation slots.
"""
restaurant_info = pd.read_excel("./data/restaurant_info.xlsx")

In [7]:
restaurant_info

,restaurant_id,restaurant_name,brief_description,restaurant_address,location,cuisine_type,seating_capacity,available_seats,contact_number,opening_hours,rating
0,R-001,Ember & Oak,Rustic-chic spot with live-fire cooking and se...,"1423 Cedar Street, Suite 200",Austin,Modern American (wood-fire),85,85,(512) 555-0142,"Mon–Sat 5 PM–11 PM, Sun 5 PM–9 PM",4.7
1,R-002,Azure Coast,"Ocean-view terrace serving fresh grilled fish,...",8801 Pacific Boulevard,Santa Monica,Coastal Mediterranean,120,120,(310) 555-0237,Daily 11 AM–10 PM,4.5
2,R-003,The Gilded Spoon,Elegant fine-dining with a tasting menu and an...,"45 Wall Street, Lobby Level",New York,Contemporary French,65,65,(212) 555-0891,Tue–Sat 6 PM–10:30 PM,4.8
3,R-004,Smoke & Stone,"Industrial-chic pitmaster joint with brisket, ...",709 Industrial Way,Portland,BBQ & Smokehouse,110,110,(503) 555-0345,Daily 11:30 AM–9 PM,4.6
4,R-005,Flora Table,"Bright, airy café sourcing 90% of produce from...",1220 Magnolia Avenue,Nashville,Plant-based / Farm-to-table,75,75,(615) 555-0678,"Mon–Fri 8 AM–8 PM, Sat–Sun 9 AM–6 PM",4.4
5,R-006,Nori & Knot,"Waterfront spot with Edomae-style sushi, robat...","3333 Alaskan Way, Pier 56",Seattle,Japanese (sushi & izakaya),95,95,(206) 555-0412,"Mon–Thu 4 PM–10 PM, Fri–Sun 12 PM–11 PM",4.9
6,R-007,Hearth & Hops,"Spacious brewpub with 24 taps, wood-fired pizz...",8745 Lincoln Highway,Denver,Gastropub / Craft beer,130,130,(303) 555-0563,"Sun–Wed 11 AM–11 PM, Thu–Sat 11 AM–1 AM",4.3
7,R-008,Saffron Sky,Bold mashup of Kerala spices and Louisiana cla...,"601 Poydras Street, Unit B",New Orleans,Indian-fusion Creole,70,70,(504) 555-0789,Tue–Sun 5 PM–10 PM,4.5
8,R-009,The Willow Jar,"Cozy, wood-clad eatery serving wild mushroom p...","2100 Main Street, Building 4",Boise,Pacific Northwest comfort,60,60,(208) 555-0123,Mon–Sat 4 PM–9:30 PM,4.2
9,R-010,Brass Lantern,Classic Beacon Hill chowder house with lobster...,42 Charles Street,Boston,New England seafood,90,90,(617) 555-0456,Daily 11:30 AM–10 PM,4.6


## Tools

#### Tool 1: search_restaurants

In [8]:
@tool
def search_restaurants(location: Optional[str] = None, cuisine_type: Optional[str] = None, min_rating: Optional[float] = None) -> list[dict]:
    """Search for restaurants using one or more optional filters.

    Use this tool when the user wants to find restaurants based on location,
    cuisine type, minimum rating, or any combination of these criteria.

    Args:
        location: City or location to search in.
        cuisine_type: Type of cuisine (e.g., Italian, Chinese, Mexican).
        min_rating: Minimum restaurant rating to include in the results.

    Returns:
        A list of matching restaurants. Each result contains the restaurant
        name, cuisine type, address, and rating.

    Notes:
        At least one search criterion must be provided.
    """
    
    if not any([location, cuisine_type, min_rating]):
        return [{"error": "At least one search criterion (location, cuisine_type, min_rating) must be provided."}]
            
    filtered_df = restaurant_info.copy()

    # Filter by location
    if location:
        filtered_df = filtered_df[
            filtered_df["location"].str.contains(
                location,
                case=False,
                na=False
            )
        ]

    # Filter by cuisine
    if cuisine_type:
        filtered_df = filtered_df[
            filtered_df["cuisine_type"].str.contains(
                cuisine_type,
                case=False,
                na=False
            )
        ]

    # Filter by minimum rating
    if min_rating is not None:
        filtered_df = filtered_df[
            filtered_df["rating"] >= min_rating
        ]

    # Keep only required columns
    filtered_df = filtered_df[
        ["restaurant_name", "cuisine_type", "restaurant_address", "rating"]
    ]

    return filtered_df.to_dict(orient="records")

In [9]:
# Test
search_restaurants.invoke(
    {
        "location": "Seattle",
        "cuisine_type": "Japanese",
        # "min_rating": 4.8
    }
)

[{'restaurant_name': 'Nori & Knot',
  'cuisine_type': 'Japanese (sushi & izakaya)',
  'restaurant_address': '3333 Alaskan Way, Pier 56',
  'rating': 4.9},
 {'restaurant_name': 'Mizu Ramen Bar',
  'cuisine_type': 'Ramen & Japanese small plates',
  'restaurant_address': '2200 Westlake Avenue',
  'rating': 4.7}]

#### Tool 2: get_restaurant_details

In [10]:
@tool
def get_restaurant_details(restaurant_name: str) -> dict:
    """
    Retrieve detailed information about a restaurant.

    Args:
        restaurant_name: The full or partial name of the restaurant.

    Returns:
        A dictionary containing:
        - restaurant_name
        - brief_description
        - restaurant_address
        - location
        - cuisine_type
        - seating_capacity
        - contact_number
        - opening_hours
        - rating

        If no restaurant is found, an error message is returned.
    """
    restaurant = restaurant_info[
        restaurant_info["restaurant_name"]
        .str.contains(restaurant_name.strip(), case=False, na=False)
    ]

    if restaurant.empty:
        return {
            "success": False,
            "message": f"No restaurant found matching '{restaurant_name}'."
        }

    return restaurant.iloc[0, 1:].to_dict()  # Return all columns except the restaurant_id column as a dictionary.

In [11]:
# Test
get_restaurant_details.invoke(
    {
        "restaurant_name": "The Gilded Spoon"
    }
)

{'restaurant_name': 'The Gilded Spoon',
 'brief_description': 'Elegant fine-dining with a tasting menu and an extensive wine cellar in a historic bank vault.',
 'restaurant_address': '45 Wall Street, Lobby Level',
 'location': 'New York',
 'cuisine_type': 'Contemporary French',
 'seating_capacity': 65,
 'available_seats': 65,
 'contact_number': '(212) 555-0891',
 'opening_hours': 'Tue–Sat 6 PM–10:30 PM',
 'rating': 4.8}

#### Tool 3: create_reservation

In [12]:
def generate_random_id():
    """
    Generate a random 6-character ID containing letters and digits.
    """
    # Combine uppercase letters, lowercase letters, and digits
    characters = string.ascii_letters + string.digits
    # Generate random 6-character string
    random_id = ''.join(random.choice(characters) for _ in range(6))
    return 'R_' + random_id


def validate_phone_number(phone_number: str, region: str = "IR") -> dict:
    """ Validate a phone number and return it in E.164 format."""
    try:
        # Parse the phone number using the specified default region.
        parsed = phonenumbers.parse(phone_number, region)
        
        # Check whether the parsed phone number is valid.
        if not phonenumbers.is_valid_number(parsed):
            return {
                "success": False,
                "error": "Invalid phone number."
            }

        # Convert the phone number to E.164 format.
        return {
            "success": True,
            "normalized_phone_number": phonenumbers.format_number(
                parsed,
                phonenumbers.PhoneNumberFormat.E164
            )
        }

    except phonenumbers.NumberParseException:
        # Raised when the phone number cannot be parsed.
        return {
            "success": False,
            "error": "Invalid phone number."
        }

In [13]:
TZ = ZoneInfo("Asia/Tehran")  # Time zone used for parsing and validating reservation dates and times

# Standard format used to store and display reservation date and time
# Example: 2026-07-05 19:30:00
DATETIME_FORMAT = "%Y-%m-%d %H:%M:%S"

In [14]:
def get_current_datetime() -> str:
    """Returns the current date and time for Iran."""
    return datetime.now(TZ).strftime(DATETIME_FORMAT)

In [15]:
def parse_reservation_datetime(user_input: str) -> dict:
    """
    Parse and validate a user-provided reservation date and time.

    The function converts a natural language date/time expression (e.g.,
    "tomorrow at 7 PM", "next Friday 18:30", "2026-07-10 19:00")
    into a standardized datetime string. It also ensures that the parsed
    datetime is valid and represents a future reservation.

    Args:
        user_input: A natural language date/time string entered by the user.

    Returns:
        A dictionary containing:
        - success (bool): Indicates whether parsing and validation succeeded.
        - datetime (str): The formatted reservation datetime if successful.
        - error (str): An error message if parsing or validation fails.
    """

    # Get the current date and time in the configured timezone
    now = datetime.now(TZ)

    # Configure the date parser
    settings = {
        # Use the current time as the reference for relative dates
        "RELATIVE_BASE": now,

        # Prefer future dates for ambiguous inputs (e.g., "Monday")
        "PREFER_DATES_FROM": "future",

        # Return timezone-aware datetime objects
        "RETURN_AS_TIMEZONE_AWARE": True,
    }

    # Parse the user's natural language date/time
    dt = dateparser.parse(user_input, settings=settings)

    # Parsing failed
    if dt is None:
        return {
            "success": False,
            "error": "Could not parse date. Enter a valid date."
        }

    # Ensure the datetime uses the configured timezone
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=TZ)
    else:
        dt = dt.astimezone(TZ)

    # Reservations cannot be made for past dates
    if dt < now:
        return {
            "success": False,
            "error": "Date must be in the future."
        }

    # Return the validated datetime in the application's standard format
    return {
        "success": True,
        "datetime": dt.strftime(DATETIME_FORMAT)
    }

**Receipt Generation**

In [16]:
is_receipt_generated = None
generated_receipt_name = None

In [17]:
def generate_receipt(reservation_number, restaurant_name, guest_name, number_of_guests, date_time, restaurant_address, restaurant_contact_number):
    """Generate a reservation receipt image containing the reservation details
    and a QR code, then save it to the Generated_Receipt directory."""
    global is_receipt_generated
    global generated_receipt_name

    # Load Receipt template
    img = Image.open('./receipt template.png')
    draw = ImageDraw.Draw(img)

    # Font
    font_title = ImageFont.truetype("C:/Windows/Fonts/times.ttf", 43)
    font_text = ImageFont.truetype("segoeui", 32)

    # Restaurant name
    restaurant_name = restaurant_name

    text_width = draw.textlength(restaurant_name, font=font_title)

    # Get image width
    image_width = img.width
    # Calculate X position
    x_position = (image_width - text_width) // 2

    # Draw centered text
    draw.text((x_position, 235), restaurant_name, fill="black", font=font_title,
        stroke_width=1, stroke_fill="black")


    # Reservation details
    draw.text((720, 578), reservation_number, fill="#800020", font=font_text, stroke_width=0.5, stroke_fill="#800020")
    draw.text((670, 660), guest_name, fill="black", font=font_text)
    draw.text((800, 745), str(number_of_guests), fill="black", font=font_text)
    draw.text((720, 835), date_time.split(' ')[0], fill="black", font=font_text)
    draw.text((750, 920), date_time.split(' ')[1], fill="black", font=font_text)
    draw.text((700, 1005), restaurant_name, fill="black", font=font_text)
    draw.text((550, 1090), restaurant_address, fill="black", font=font_text)
    draw.text((680, 1170), restaurant_contact_number, fill="black", font=font_text)


    # Get QR Code
    qr = qrcode.QRCode(version=2, error_correction=qrcode.constants.ERROR_CORRECT_L,
                        box_size=6, border=2)
    qr.add_data(f"RESERVATION: {reservation_number}")
    qr.make(fit=True)
    qr_img = qr.make_image(fill_color="black", back_color="white")
    qr_img = qr_img.resize((200, 200), Image.Resampling.LANCZOS)

    img.paste(qr_img, ((image_width+280)//2, 1270))

    # Save
    receipt_name = f"{reservation_number}_reservation_receipt.png"
    img.save(f"Generated_Receipt/{receipt_name}")

    # Change is_receipt_generated to True, and get receipt name
    is_receipt_generated = True
    generated_receipt_name = receipt_name

In [18]:
def make_a_reservation(restaurant_id, customer_name, number_of_guests, customer_contact_number, reservation_datetime):
    """Create and store a restaurant reservation.

    This function updates the restaurant's available seating capacity,
    records the reservation in the customer_reservations.xlsx file,
    and returns the reservation details."""
    
    # Decrease the available seating capacity of the restaurant
    restaurant_info.at[restaurant_info[restaurant_info['restaurant_id'] == restaurant_id].index[0], 'available_seats'] -= number_of_guests

    # Select the restaurant
    restaurant = restaurant_info.loc[restaurant_info["restaurant_id"] == restaurant_id].iloc[0]
    
    # Timestamp when the reservation was created or registered
    reservation_timestamp = get_current_datetime()

    # Add reservation information to an Excel file
    ##  1. Read the Excel file
    customer_reservations = pd.read_excel("./data/customer_reservations.xlsx")
    ## 2. Add new row
    new_reservation = {'reservation_number': generate_random_id(), 'restaurant_id': restaurant_id,
                        'restaurant_name': restaurant["restaurant_name"],
                        'restaurant_address': restaurant["restaurant_address"],
                        'location': restaurant["location"],
                        'restaurant_contact_number': restaurant["contact_number"],
                        'customer_name': customer_name,
                        'number_of_guests': number_of_guests,
                        'customer_contact_number': customer_contact_number,
                        'reservation_datetime': reservation_datetime,
                        'reservation_created_at': reservation_timestamp}
    
    ## 3. Add to dataframe
    customer_reservations.loc[len(customer_reservations)] = new_reservation

    ## 4. Save to Excel file
    customer_reservations.to_excel('./data/customer_reservations.xlsx', index=False)

    # Return reservation information except the restaurant_id
    new_reservation.pop('restaurant_id')
    return new_reservation


@tool
def create_reservation(restaurant_name: str, customer_name:str, number_of_guests: int, customer_contact_number: str, reservation_datetime: str) -> dict:
    """Create a reservation for a customer at a restaurant.

Use this tool only after the user has selected a restaurant and has provided
all required reservation information.

Required information:
- restaurant_name: The name of the restaurant.
- customer_name: The name under which the reservation will be made.
- number_of_guests: The total number of guests. Must be greater than 0.
- customer_contact_number: A phone number that the restaurant can use to contact the customer.
- reservation_datetime: The reservation date and time in a valid format (for example,
  '2026-06-30 19:30' or another format supported by the application).

This tool checks whether the restaurant has enough available seats. If sufficient
capacity exists, it creates the reservation, updates the restaurant's available
seating capacity, stores the reservation information, and returns the reservation
details.

Returns:
    A dictionary containing:
    - reservation_status: "successful" or "failed".
    - message: A human-readable description of the result.
    - reservation: Reservation details if the reservation was created successfully.
"""
    restaurant = restaurant_info[
        restaurant_info["restaurant_name"]
        .str.contains(restaurant_name.strip(), case=False, na=False)
    ]


    # Check if the mobile/phone number is valid or not
    contact_number_status = validate_phone_number(customer_contact_number)
    if contact_number_status['success'] == False:
        return {'error': contact_number_status['error']}
    else:
        customer_contact_number = contact_number_status['normalized_phone_number']  # e.g. +98911......

    # Check whether the time is valid or not
    reservation_date_status = parse_reservation_datetime(reservation_datetime)
    if reservation_date_status['success'] == False:
        return {'error': reservation_date_status['error']}
    else:
        reservation_datetime = reservation_date_status['datetime']
    
    # Get number of available steats
    available_seats = int(restaurant['available_seats'].iloc[0])
    restaurant_id = restaurant['restaurant_id'].iloc[0]

    # Checking whether there are enough seats for the number of guests.
    if number_of_guests <= 0:
        return {"reservation_status": "failed", "message": "The number of guests cannot be 0 or negative. Please enter a valid number."}
    if available_seats == 0:
        return {"reservation_status": "failed", "message": "Unfortunately, this restaurant has no available seats."}
    elif number_of_guests <= available_seats:
        reservation_info = make_a_reservation(restaurant_id, customer_name, number_of_guests, customer_contact_number, reservation_datetime)
        
        generate_receipt(
            reservation_number=reservation_info['reservation_number'],
            restaurant_name=reservation_info['restaurant_name'],
            guest_name=customer_name,
            number_of_guests=number_of_guests,
            date_time=reservation_datetime,
            restaurant_address=reservation_info['restaurant_address'],
            restaurant_contact_number=reservation_info['restaurant_contact_number']
            )
       
        return {
                "reservation_status": "successful",
                "message": "Reservation created successfully. Your receipt has been printed.",
                "reservation": reservation_info
                }
    elif number_of_guests > available_seats:
        return {"reservation_status": "failed", "message": f"The number of guests exceeds the current capacity of this restaurant. Number of available seats: {available_seats}"}
    
    else:
        return {"message": "An unpredictable situation arose."}

In [19]:
restaurant_info.head(2)

,restaurant_id,restaurant_name,brief_description,restaurant_address,location,cuisine_type,seating_capacity,available_seats,contact_number,opening_hours,rating
0,R-001,Ember & Oak,Rustic-chic spot with live-fire cooking and se...,"1423 Cedar Street, Suite 200",Austin,Modern American (wood-fire),85,85,(512) 555-0142,"Mon–Sat 5 PM–11 PM, Sun 5 PM–9 PM",4.7
1,R-002,Azure Coast,"Ocean-view terrace serving fresh grilled fish,...",8801 Pacific Boulevard,Santa Monica,Coastal Mediterranean,120,120,(310) 555-0237,Daily 11 AM–10 PM,4.5


In [ ]:
# test
create_reservation.invoke(
 {
     'restaurant_name': "Azure Coast",
     'customer_name': 'Alex Wood',
     'number_of_guests': 5,
     'customer_contact_number': '09111234567',
     'reservation_datetime': '9th April at 5pm'
 }   )

#### Tool 4: answer_restaurant_assistant_faq

In [20]:
# Load FAQs dataset
restaurant_assistant_faq = pd.read_excel('./data/restaurant_assistant_faq.xlsx')
display(restaurant_assistant_faq.head())

# Building Document
documents = []
for i, row in restaurant_assistant_faq.iterrows():
    question = row['question']
    answer = row['answer']
    
    doc = Document(
        page_content=question,
        metadata={"answer": answer},
    )
    documents.append(doc)

,question,answer
0,How do I make a reservation?,You can make a reservation by providing the re...
1,Can I reserve a table for today?,"Yes, reservations can be made for the same day..."
2,How can I check if a restaurant has available ...,You can ask for availability by specifying the...
3,Can I reserve a table without a phone number?,No. A valid contact number is required to comp...
4,Is there a fee for making a reservation?,No. Reservations are free unless the restauran...


In [21]:
# Vector Store
from langchain_chroma import Chroma

# Chroma stores its data in the directory specified by `persist_directory`. When you create a `Chroma` object, it checks whether the collection already exists in that directory:
# * If the collection exists → Chroma opens the existing collection.
# * If the collection does not exist → Chroma creates a new collection.

if not os.path.exists("./chroma_langchain_db"):
    # Build index once
    vector_store = Chroma(
    collection_name="restaurant_assistant_faq",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
    )
    ids = vector_store.add_documents(documents=documents, ids=['q_'+str(i) for i in range(len(documents))])

else:
    # Load existing index
    vector_store = Chroma(
    collection_name="restaurant_assistant_faq",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
    )

print("Number of documents in vectorstore:", vector_store._collection.count())

Number of documents in vectorstore: 35


In [22]:
# Retriever
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)


def build_context(question):
    """Build a retrieval context for the user's question.

    The function retrieves the most relevant FAQ documents from the vector
    store and formats them into a context string that can be passed to an
    LLM for Retrieval-Augmented Generation (RAG).
    
    Args:
        question: The user's natural language question.

    Returns:
        A formatted context string containing the user's question and the
        retrieved question-answer pairs."""
    
    retrieved_docs = retriever.batch([question])[0]
    relevant_qa = ''
    for doc in retrieved_docs:
        relevant_qa += 'Question: ' + doc.page_content + '\nAnswer: ' + doc.metadata['answer'] + '\n---\n'


    context = f"""USER QUESTION: {question}


RELEVANT QUESTIONS AND ANSWERS:

    {relevant_qa}
    """
    return context

In [23]:
@tool
def answer_restaurant_assistant_faq(question: str) -> dict:
    """
    Answer general FAQ questions about the restaurant reservation system.
    Use this tool for questions about reservation policies.
    
    Do not use this tool for searching restaurants, checking availability,
    creating reservations, or managing existing reservations.
    If you do not find the answer to the user's question in the provided context, reply: "I don't know the answer to your question."
    Args:
        question: The user's question.

    Returns:
        A relevant answer from the FAQ knowledge base.
    """
    # Get relevant context for answering the question
    context = build_context(question)
    return {"relevant_context_to_answer_user_question": context}

In [24]:
# Test
print(answer_restaurant_assistant_faq.invoke({
    "question": "How can I book a reservation?"
})['relevant_context_to_answer_user_question'])

USER QUESTION: How can I book a reservation?


RELEVANT QUESTIONS AND ANSWERS:

    Question: How do I make a reservation?
Answer: You can make a reservation by providing the restaurant name, number of guests, reservation date, and your contact information.
---
Question: What information do I need before making a reservation?
Answer: You need the restaurant name, reservation date, guest count, customer name, and contact number.
---
Question: How do I cancel a reservation?
Answer: Provide your reservation number, and the reservation can be cancelled.
---

    


#### Tool 5: cancel_reservation

In [25]:
@tool
def cancel_reservation(reservation_number: str) -> dict:
    """Cancel a reservation by reservation number."""
    
    # Open reservation data file
    customer_reservations = pd.read_excel("./data/customer_reservations.xlsx")
    
    # Check if reservation code exists in file
    if reservation_number not in customer_reservations['reservation_number'].values:
        return {"message": f"The reservation number you provided ({reservation_number}) does not exist. Please verify that you have entered the correct reservation number."}
    else: 
        # Get restaurant id
        restaurant_id = customer_reservations[customer_reservations['reservation_number']==reservation_number]['restaurant_id'].iloc[0]

        # Get number of guests
        number_of_guests = int(customer_reservations.loc[customer_reservations['reservation_number'] == reservation_number,'number_of_guests'].iloc[0])

        # Increase the available seating capacity of the restaurant
        restaurant_info.at[restaurant_info[restaurant_info['restaurant_id'] == restaurant_id].index[0], 'available_seats'] += number_of_guests

        # Delete row by reservation number
        customer_reservations = customer_reservations[customer_reservations["reservation_number"] != reservation_number].reset_index(inplace=False, drop=True)
       
        # Update the Excel file
        customer_reservations.to_excel('./data/customer_reservations.xlsx', index=False)
        
        return {"message": f"Reservation **{reservation_number}** is cancelled."}

#### list of the tools
Give the model a list of the tools you want to use:

In [26]:
# tools list
tools = [search_restaurants,
        get_restaurant_details,
        create_reservation, 
        answer_restaurant_assistant_faq, 
        cancel_reservation]


In [27]:
system_prompt = """You are a professional Restaurant Reservation Assistant.

Your primary responsibility is to help users find restaurants, view restaurant information, create reservations, and answer questions about the restaurant reservation system.

You have access to specialized tools. Use them whenever restaurant information, reservation data, or FAQ knowledge is required. Do not invent restaurant details, availability information, ratings, reservation numbers, or policies. Always rely on tool results.

Available capabilities:

Search restaurants by location, cuisine type, and rating.
Retrieve restaurant details.
Create restaurant reservations.
Cancel a reservation.
Answer frequently asked questions about the reservation system.

Guidelines:

"search_restaurants":
Use the restaurant search tool when users ask to find restaurants.
Apply all preferences provided by the user.
If information is missing, ask a concise follow-up question.

"get_restaurant_details":
Use the restaurant details tool when the user asks about a specific restaurant.
Provide only information returned by the tool.

"create_reservation":
Before creating a reservation, collect all required information:
Restaurant name
Customer name
Number of guests
Contact number
Reservation date
If any required information is missing, ask for it before calling the reservation tool.
After a successful reservation, clearly present the reservation details especially the reservation number..
If the reservation cannot be completed, explain the reason provided by the tool.

"answer_restaurant_assistant_faq":
Use the FAQ tool for questions about reservation policies, cancellations, ratings, booking requirements, and other general system questions.
Do not answer policy questions from your own knowledge if an FAQ tool is available.

Communication Style:
- Always respond in a highly professional, respectful, and courteous manner. Use clear, concise, and customer-friendly language, and maintain a polite tone throughout the conversation.
- Do not expose internal tool names, database structure, source files, or implementation details.
- Do not make assumptions about missing information.

Error Handling:
- If a tool returns no results, inform the user politely and suggest alternatives when appropriate.
- If a request is outside the scope of restaurant search, restaurant information, reservations, or FAQs, politely explain that you can only assist with restaurant-related requests.

Always prioritize accuracy over speed and use tools whenever external information is required.
"""

## Run

In [28]:
agent = create_agent(
    model="openrouter:gemini-2.5-flash-lite",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt=system_prompt,
    name="restaurant_reservation_assistant"
)


In [29]:
# Clear history
config = {"configurable": {"thread_id": str(uuid7())}}

In [30]:
config

{'configurable': {'thread_id': '019f4342-403b-7703-8258-c89cf9a9de9a'}}

## Test with UI

In [31]:
import gradio as gr

def ask_question(question, history):
    global is_receipt_generated
    is_receipt_generated = False  # flag

    result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config=config,
    )

    # Show the chatbot response with a receipt image
    if is_receipt_generated == True:
        return [result['messages'][-1].content, '\nYour Receipt:\n', gr.Image('Generated_Receipt/'+generated_receipt_name)]
    # If the receipt is not printed, just show the chatbot response.
    else:
        return result['messages'][-1].content

In [ ]:
# Create the Gradio chat interface
gr.ChatInterface(
    # Function that processes each user message and returns the assistant's response
    ask_question,

    # Chat window for displaying the conversation
    chatbot=gr.Chatbot(height=600),

    # User input textbox
    textbox=gr.Textbox(
        placeholder="Ask your question...",  # Placeholder text
        container=False,                     # Remove the surrounding container
        scale=7,                             # Width relative to other components
        submit_btn=True                      # Show the submit button
    ),

    # Title displayed at the top of the interface
    title="Restaurant Reservation Assistant 🍽️",

    # Short description displayed below the title
    description="Your Restaurant Reservation Assistant",

    # Cache example inputs (if examples are provided)
    cache_examples=True,

# Launch the web application
).launch(
    theme="monochrome", 
    share=False           # False: Do not create a public sharing link
)

Close all currently running Gradio interfaces to release resources

In [ ]:
gr.close_all() 

<div style="
    background:linear-gradient(135deg,#9941d8,#5b86e5);
    color:white;
    padding:25px;
    border-radius:15px;
    text-align:center;
    margin-top:20px;
">

<h2 style="margin:0;">🎉 End of Notebook!</h2>


<p>
Thank you for reviewing this project.
</p>

<p>
<b>Restaurant Reservation System</b><br>

<p>
Created by <b>MohammadReza</b> - 2026
</p>


</div>